In [1]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.shape

(50000, 2)

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

# Pre-processing

In [8]:
#1. convert to lowercase
df["review"] = df["review"].str.lower()

In [9]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [10]:
#2. remove URLs
def remove_urls(text):
    text = re.sub(r"http\S+","",text)
    return text
    
df["review"] = df["review"].apply(remove_urls)

In [11]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [12]:
#3. remove html tags
def remove_html_tags(text):
    text = re.sub(r"<.*?>","",text)
    return text
    
df["review"] = df["review"].apply(remove_html_tags)  

In [13]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [14]:
#4. remove punctuations
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text)
    return text
    
df["review"] = df["review"].apply(remove_punctuations)  

In [15]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [16]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /Users/suhani/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/suhani/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/suhani/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [17]:
#5. Remove stopwords
def remove_stopwords(text):
    words = word_tokenize(text)
    stop_words = stopwords.words("english")
    filtered_words = []
    
    for word in words:
        if word not in stop_words:
            filtered_words.append(word)
    return " ".join(filtered_words)

df["review"] = df["review"].apply(remove_stopwords)

In [18]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


In [19]:
#6. Stemming
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [20]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,wonder littl product film techniqu unassum old...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


In [21]:
#7. Encoding
le = LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])

In [22]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,1
1,wonder littl product film techniqu unassum old...,1
2,thought wonder way spend time hot summer weeke...,1
3,basic there famili littl boy jake think there ...,0
4,petter mattei love time money visual stun film...,1


# Vectorization

In [23]:
tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["review"])
y = df["sentiment"]

# Train-Test Split

In [24]:
X_train , X_test, y_train, y_test = train_test_split(
    X,y,test_size = 0.2, random_state=42
)

In [25]:
X_train = X_train.toarray()
X_test = X_test.toarray()

# Dataset

In [26]:
train_dataset = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_dataset = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

# DataLoader

In [27]:
train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=64,)

# Build RNN

In [28]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #RNN Layer
        self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first=True)
        # Fully connected Layer
        self.fc = nn.Linear(hidden_size,1)

    def forward(self,x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [29]:
input_size = X_train.shape[1]
model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# Training The RNN

In [30]:
epochs = 10
for epoch in range(epochs):
    model.train()
    for Xb,yb in train_loader:
        optimizer.zero_grad()
        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)
        outputs = torch.sigmoid(outputs.squeeze())
        loss = criterion(outputs,yb)
        loss.backward()
        optimizer.step()
    print(f"epoch = {epoch}/{epochs} , loss = {loss.item()}")
    

epoch = 0/10 , loss = 0.3233710825443268
epoch = 1/10 , loss = 0.230363667011261
epoch = 2/10 , loss = 0.2734505534172058
epoch = 3/10 , loss = 0.2687605619430542
epoch = 4/10 , loss = 0.30398067831993103
epoch = 5/10 , loss = 0.21777386963367462
epoch = 6/10 , loss = 0.11084754765033722
epoch = 7/10 , loss = 0.13044776022434235
epoch = 8/10 , loss = 0.19786140322685242
epoch = 9/10 , loss = 0.21583028137683868


# Evaluate Model

In [31]:
model.eval()
with torch.no_grad():
    correct_vals = 0
    total_vals = 0
    for Xb,yb in test_loader:
        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze())>0.5).float()
        total_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()
    print(f"accuracy = {correct_vals/total_vals*100}")
        

accuracy = 87.06261974387417
